In [ ]:
# Run this cell if you're using from colab
#!git clone https://github.com/R-Oc-A/HackathonPastryLPV.git
#!wget https://github.com/R-Oc-A/HackathonPastryLPV/releases/download/IntensityGrids/grids.tar.gz
#!tar -xvf grids.tar.gz
#!pip install https://github.com/R-Oc-A/HackathonPastryLPV/releases/download/wheel/pastrypy-010-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
#!pip install tomli_w
#import sys
#sys.path.append('/content/HackathonPastryLPV')
#import os
#os.environ["GRIDS"]="/content/ema_parquets/"

In [ ]:
import pastrypy as psp
import pulsation_description as plsd
import line_profile_description as lpd
import tomli_w
import os
import polars as pl
import matplotlib.pyplot as plt

# Pulstar configuration
Here you specify the star you'll be modelling as well as the modes of pulsation

In [ ]:
#Taken from a Simbad quick Query and from Teltings paper
mode=plsd.Mode(l=2,m=1,
                rel_dr=0.0024,
                k=0.03,frequency=5.38,
                phase_offset=0.0,
                rel_dtemp=2.62,
                phase_rel_dtemp=180.0,
                rel_dg=10.0,
                phase_rel_dg=34.0,
                rotation_effects = "PerturbativeCoriolis")
star_data=plsd.StarData(mass=10.0,
                        radius=6.93,
                        effective_temperature=21642.0,
                        v_omega=20.0,
                        inclination_angle=45.0)
time_points=plsd.TimePoints(Uniform=plsd.UniformTime(start=0.0,end=0.0,step=0.01))
#mesh=plsd.Mesh(Sphere=plsd.SphericalStar(theta_step=2.0,phi_step=4.0))
mesh=plsd.Mesh(HSphere=plsd.HealpixStar(depth=3))

pulsconfig=plsd.PulstarConfig(mode_data=[mode],star_data=star_data,time_points=time_points,mesh=mesh)
pulsconfig_dict=pulsconfig.model_dump(exclude_none=True)
puls_toml_string=tomli_w.dumps(pulsconfig_dict)

# Profile configuration
Here you specify the line profile variability you want to observe.

In [ ]:
#Taken from a Simbad quick Query
wl_range=lpd.WavelengthRange(start=4551.0,end=4555.0,step=0.0033)
#path_to_grids="../profile/grids/"
path_to_grids = os.getenv("GRIDS")
#path_to_grids = f"{os.getenv("GRIDS")}ema_parquets/"
#print(path_to_grids)
grid1=lpd.IntensityGrid(EmaParquet=lpd.EmaGrid(temperature=20000.0,log_gravity=3.5,metalicity=0.0,filename="lp0000_20000_0350_0020.parquet"),Nadya=None)
grid2=lpd.IntensityGrid(EmaParquet=lpd.EmaGrid(temperature=20000.0,log_gravity=3.8,metalicity=0.0,filename="lp0000_20000_0380_0020.parquet"),Nadya=None)
grid3=lpd.IntensityGrid(EmaParquet=lpd.EmaGrid(temperature=24000.0,log_gravity=3.5,metalicity=0.0,filename="lp0000_24000_0350_0020.parquet"),Nadya=None)
grid4=lpd.IntensityGrid(EmaParquet=lpd.EmaGrid(temperature=24000.0,log_gravity=3.8,metalicity=0.0,filename="lp0000_24000_0380_0020.parquet"),Nadya=None)
prof_config=lpd.ProfileConfig(max_velocity=1.0e2,path_to_grids=path_to_grids,wavelength_range=wl_range,intensity_grids=[grid1,grid2,grid3,grid4])

prof_config_dict=prof_config.model_dump(exclude_none=True)

prof_toml_string=tomli_w.dumps(prof_config_dict)
print(prof_toml_string)

# First run

In [ ]:
pulse_df = psp.pulstar(puls_toml_string)

In [ ]:
pulse_df.sort("area").tail(5)

In [ ]:
wavelength_df = psp.profile(prof_toml_string,pulse_df)

In [ ]:
wavelength_df.head(5)

In [ ]:
#Taken from a Simbad quick Query and from Teltings paper
nonrot= plsd.NonRot()
pertcor=plsd.PerturbCor()
tar = plsd.TAR()
cendef = plsd.CenDef(coefficient_expansion=[-0.856,0.01,0.0])
rotation_regime = plsd.RotationRegime(NonRotating=None,PerturbativeCoriolis=None,Tar=None,CentrifugalDeformation=None)
#rotation_regime.NonRotating=nonrot
#rotation_regime.PerturbativeCoriolis=pertcor
#rotation_regime.Tar=tar
rotation_regime.CentrifugalDeformation=cendef

mode=plsd.Mode(l=2,m=1,
                rel_dr=0.0024,
                k=0.03,frequency=5.38,
                phase_offset=0.0,
                rel_dtemp=2.62,
                phase_rel_dtemp=180.0,
                rel_dg=10.0,
                phase_rel_dg=34.0,
                rotation_effects = rotation_regime)
star_data=plsd.StarData(mass=10.0,
                        radius=6.93,
                        effective_temperature=21642.0,
                        v_omega=20.0,
                        inclination_angle=45.0)
time_points=plsd.TimePoints(Uniform=plsd.UniformTime(start=0.0,end=0.0,step=0.01))
#mesh=plsd.Mesh(Sphere=plsd.SphericalStar(theta_step=2.0,phi_step=4.0))
mesh=plsd.Mesh(HSphere=plsd.HealpixStar(depth=3))

pulsconfig=plsd.PulstarConfig(mode_data=[mode],star_data=star_data,time_points=time_points,mesh=mesh)
pulsconfig_dict=pulsconfig.model_dump(exclude_none=True)
puls_toml_string=tomli_w.dumps(pulsconfig_dict)
print(puls_toml_string)